In [1]:
from datetime import date

import pandas as pd
from sqlalchemy import func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons (applicants and inventors)
    TLS207_PERS_APPLN,          # person <-> application links
    TLS209_APPLN_IPC,           # IPC classification
    TLS224_APPLN_CPC,           # CPC classification
    TLS231_INPADOC_LEGAL_EVENT, # legal events
)

In [2]:
patstat = PatstatClient(env='TEST')
db = patstat.orm()

In [14]:
FLYING_DRONES_KEYWORDS = (
    "unmanned aerial vehicle",
    "UAV",
    "unmanned aircraft system",
    "UAS",
    "remotely piloted aircraft system",
    "RPAS",
    "remotely piloted vehicle",
    "RPV",
    "uncrewed aerial vehicle",
    "autonomous aerial vehicle",
    "quadcopter",
    "multirotor",
    "multicopter",
    "hexacopter",
    "octocopter",
    "tricopter",
    "single-rotor UAV",
    "fixed-wing drone",
    "fixed-wing UAV",
    "vertical takeoff and landing",
    "VTOL",
    "tilt-rotor aircraft",
    "hybrid VTOL",
    "micro air vehicle",
    "MAV",
    "nano drone",
    "FPV drone",
    "first-person view drone",
    "tethered drone",
    "flying platform",
    "ornithopter",
    "aerostat drone",
    "flight control system",
    "flight controller",
    "autopilot system",
    "automatic pilot",
    "attitude control",
    "position control",
    "altitude hold",
    "heading hold",
    "flight dynamics",
    "roll pitch yaw control",
    "electronic speed controller",
    "ESC",
    "brushless motor",
    "thrust vectoring",
    "rotor assembly",
    "propeller assembly",
    "airframe",
    "gimbal mount",
    "payload deployment",
    "payload bay",
    "landing gear",
    "alighting gear",
    "foldable rotor arm",
    "deployable wings",
    "autonomous navigation",
    "waypoint navigation",
    "flight path planning",
    "trajectory planning",
    "obstacle avoidance",
    "collision avoidance",
    "sense and avoid",
    "simultaneous localization and mapping",
    "visual SLAM",
    "geo-fencing",
    "return-to-home",
    "RTH",
    "autonomous landing",
    "precision landing",
    "ground control station",
    "GCS",
    "unmanned traffic management",
    "UTM",
    "aerial traffic control",
    "remote identification",
    "Remote ID",
    "telemetry link",
    "drone swarm",
    "swarm robotics",
    "multi-UAV coordination",
    "cooperative flight",
    "formation flying",
    "aerial imaging",
    "aerial photography",
    "aerial surveillance",
    "airborne sensor platform",
    "drone delivery",
    "parcel delivery UAV",
    "aerial inspection",
    "agricultural spraying drone",
    "tethered power supply",
    "docking station",
    "charging pad",
    "automatic battery swapping",
    "wireless charging dock"
)

In [15]:
FLYING_DRONES_IPC_CODES = (
    "B64U",        # Unmanned Aerial Vehicles [UAV]; Equipment therefor (Primary dedicated CPC subclass)
    "B64U10/00",   # UAV types (Rotorcraft, VTOL, Fixed-wing, MAV, Tethered)
    "B64U20/00",   # Constructional aspects (Modular, Foldable, Stealth, Airframe)
    "B64U30/00",   # Lift-producing means (Rotors, Rotor guards, Wings, Shrouded rotors)
    "B64U40/00",   # Mechanical control surface and rotor adjustment mechanisms
    "B64U50/00",   # Power supply, propulsion, motor transmission, in-flight charging
    "B64U60/00",   # Ground-engaging elements, rolling cages, landing gear
    "B64U70/00",   # Launching, take-off, landing, platforms, arresting gear
    "B64U80/00",   # Transport, storage, and handling systems for UAVs
    "B64U2101/00", # Indexing codes for specific applications (Delivery, Agriculture, Rescue, Surveillance)
    "B64C39/02",   # Legacy CPC code for Unmanned Aircraft / Remotely Piloted Vehicles (Pre-2023 filings)
    "B64C27/08",   # Multi-rotor rotorcraft (Quadcopters, Hexacopters)
    "G05D1/00",    # Control of position, course, altitude, attitude (Autopilot, Flight Control)
    "G05D1/02",    # Control of position or course in 2D space (Obstacle avoidance, Trajectory)
    "G05D1/10",    # Control of position or course in 3D space (Autonomous flight pathing)
    "G08G5/00",    # Traffic control of aircraft / Unmanned Traffic Management (UTM)
    "G08G5/04",    # Anti-collision and sense-and-avoid systems
    "B64D47/00",   # Equipment fitted in or to aircraft (Payloads, Camera gimbals, Dropping mechanisms)
    "B64F1/00",    # Ground support, launching catapults, docking stations, charging pads
    "H04B7/185",   # Radio communication, telemetry, and control links for airborne vehicles
)

In [16]:
FLYING_DRONES_CPC_CODES = (
     "B64U",        # Unmanned Aerial Vehicles [UAV]; Equipment therefor (Adopted in IPC taxonomy)
    "B64C39/02",   # Unmanned aircraft, remotely piloted vehicles (Standard historical IPC code)
    "B64C27/00",   # Rotorcraft; Rotors peculiar to rotorcraft
    "B64C27/08",   # Rotorcraft with two or more rotors (Multirotors)
    "G05D1/00",    # Automatic pilot and flight control systems
    "G05D1/02",    # Navigation and path guidance in 2D
    "G05D1/10",    # Navigation and 3D path control
    "G08G5/00",    # Air traffic control systems
    "B64D47/00",   # Equipment fitted to aircraft (Sensors, Payloads)
    "B64F1/00",    # Ground installations and launching/landing equipment
    "H04B7/185",   # Wireless telemetry and communication networks for aircraft
)

In [17]:
def keyword_condition(terms):
    """OR-condition: any term appears in the title or in the abstract (case-insensitive)."""
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(
        *(title.like(f"%{t.lower()}%") for t in terms),
        *(abstract.like(f"%{t.lower()}%") for t in terms),
    )


def symbol_prefix_condition(symbol_column, codes):
    """OR-condition: the blank-stripped classification symbol starts with one of the codes."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


# Base columns shown for every application
APPLN_COLUMNS = (
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_auth,
    TLS201_APPLN.appln_nr,
    TLS201_APPLN.appln_kind,
    TLS201_APPLN.appln_filing_date,
)


def query_by_keywords(*columns):
    """Applications whose title or abstract contains an AI keyword."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(keyword_condition(FLYING_DRONES_KEYWORDS))
    )


def query_by_ipc(*columns):
    """Applications with at least one IPC symbol in the AI list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
        .filter(symbol_prefix_condition(TLS209_APPLN_IPC.ipc_class_symbol, FLYING_DRONES_IPC_CODES))
    )


def query_by_cpc(*columns):
    """Applications with at least one CPC symbol in the AI list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, FLYING_DRONES_CPC_CODES))
    )

### KEYWORD SEARCH

In [ ]:
q_keywords = query_by_keywords(
    *APPLN_COLUMNS,
    TLS202_APPLN_TITLE.appln_title,
    TLS203_APPLN_ABSTR.appln_abstract,
).distinct()

df_keywords = patstat.df(q_keywords)
df_keywords.head(10)

### IPC SEARCH

In [19]:
q_ipc = query_by_ipc(*APPLN_COLUMNS, TLS209_APPLN_IPC.ipc_class_symbol).distinct()

df_ipc = patstat.df(q_ipc)
df_ipc.head()

,appln_id,appln_auth,appln_nr,appln_kind,appln_filing_date,ipc_class_symbol
0,52104346,US,59725632,A,1932-03-07,B64C 39/02
1,51203027,US,47874943,A,1943-03-11,B64C 39/02
2,51780868,US,55377755,A,1955-12-19,G05D 1/00
3,53693756,US,83006959,A,1959-07-28,B64C 39/02
4,49770813,US,3426982D,A,1967-02-03,B64C 39/02


### CPC SEARCH

In [20]:
q_cpc = query_by_cpc(*APPLN_COLUMNS, TLS224_APPLN_CPC.cpc_class_symbol).distinct()

df_cpc = patstat.df(q_cpc)
df_cpc.head(10)

,appln_id,appln_auth,appln_nr,appln_kind,appln_filing_date,cpc_class_symbol
0,52104346,US,59725632,A,1932-03-07,B64C 39/022
1,52104346,US,59725632,A,1932-03-07,B64U 10/60
2,51203027,US,47874943,A,1943-03-11,B64C 39/02
3,51888038,US,56784244,A,1944-12-12,B64C 27/001
4,48129109,US,14298850,A,1950-02-08,B64C 27/008
5,49451273,US,30378852,A,1952-08-11,B64C 27/08
6,51780868,US,55377755,A,1955-12-19,G05D 1/0061
7,53693756,US,83006959,A,1959-07-28,B64C 39/02
8,18848864,FR,845407,A,1960-11-30,B64C 27/008
9,50819257,US,42940865,A,1965-02-01,B64C 27/008


### COMBINE KEYWORDS AND IPC AND CPC

In [21]:
appln_id = TLS201_APPLN.appln_id.label("appln_id")

ai_ids = (
    query_by_keywords(appln_id)
    .union(query_by_ipc(appln_id), query_by_cpc(appln_id))
    .subquery()
)

q_ai = (
    db.query(*APPLN_COLUMNS, TLS202_APPLN_TITLE.appln_title)
    .select_from(TLS201_APPLN)
    .join(ai_ids, TLS201_APPLN.appln_id == ai_ids.c.appln_id)
    .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
    .distinct()
    .order_by(TLS201_APPLN.appln_filing_date)
)

df_ai = patstat.df(q_ai)
df_ai.head(10)

,appln_id,appln_auth,appln_nr,appln_kind,appln_filing_date,appln_title
0,23418246,GB,189803666D,A,1898-02-14,Improvements relating to Screw-anchors or Scre...
1,23534871,GB,190613963D,A,1906-06-18,Improvements in Conveyors.
2,47101266,US,1906346330,A,1906-12-04,FIRE-ESCAPE.
3,47463534,US,1910545146,A,1910-02-21,FIRE-ESCAPE.
4,23600238,GB,191012889D,A,1910-05-27,Improvements in the Method of Cooling the Comm...
5,23608713,GB,191028025D,A,1910-12-02,None
6,47497746,US,1911638270,A,1911-07-13,FIRE-ESCAPE.
7,23626695,GB,191129058D,A,1911-12-23,Improvements in or relating to Aeroplanes.
8,23645427,GB,191301510D,A,1913-01-18,Improvements in Shock Absorbing Devices for Mo...
9,23639662,GB,191221705D,A,1913-06-27,Improvements in and connected with Fluid Press...


In [22]:
# How much does each strategy contribute? (counts of distinct applications)
pd.Series(
    {
        "keywords": df_keywords["appln_id"].nunique(),
        "IPC": df_ipc["appln_id"].nunique(),
        "CPC": df_cpc["appln_id"].nunique(),
        "union (AI set)": df_ai["appln_id"].nunique(),
    },
    name="applications",
)

keywords          58217
IPC                2809
CPC                3516
union (AI set)    60458
Name: applications, dtype: int64